# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manalchaudharyy/FlyrankAI-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
**Task type: Scoring / Ranking, built on a binary classification core**

The final output for my lane is a **ranked review queue** — the Lane 2 spec itself says so
("Ranked review queue with scores, actions, and reason codes"). But under the hood, the
score that drives the ranking comes from a **binary classification** model: is this page
"declining" (1) or not (0)? I don't need the model to output a class label directly — I need
its predicted *probability*, which becomes the ranking score. So this is not clustering
(I'm not grouping pages into unlabeled types) and not pure ranking (I do have a defined
target, not just relative comparisons) — it's classification whose output gets consumed as
a score for prioritization.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?***Target/proxy:** `is_declining_label = (trend_direction == "down")`

This label comes from a **defined rule applied to observed data**, not a raw future outcome —
it's a proxy. It buckets the *current* 90-day window into up/flat/down, rather than asking
"will this page decline over the *next* 30 days." That's a known weakness the lane guide
calls out directly: a stronger capstone target would be a future-window label
(`prior 90 days of features -> decline over next 30 days`), which needs a clean feature/target
split with no leakage across the window boundary. I'm starting with the current-window proxy
because it's what the starter dataset ships and lets me validate the whole pipeline end to
end; I'll revisit a future-window version once I have warehouse access with real daily
timestamps (Week 3+).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*
**Metric: Precision@50** (with Precision@20 as a secondary check)

I'm not optimizing for accuracy or even ROC AUC as the headline number, because a reviewer
doesn't read 30,000 predictions — they read the top of a list, sized to their real capacity
(e.g. 50 pages/review cycle). Precision@50 asks exactly the question that matters for the
decision: "of the top 50 pages this queue tells me to review first, how many are actually
declining?" That's a defensible number because it's tied directly to the action someone
takes (Section 2 from last week's notebook), not an abstract score. I'll also look at
Precision@20 since capacity may be tighter some cycles.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/manalchaudharyy/FlyrankAI-ML"
REPO_DIR = "FlyrankAI-ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Unique content_id count: {df['content_id'].nunique()}  <- confirms one row = one page")
print(f"Declining rate (target balance): {df['is_declining_label'].mean():.1%}\n")

# The unit of analysis, as a real dataframe: one row = one page
cols_to_show = ["content_id", "impressions_90d", "days_since_last_update",
                 "avg_position", "ctr", "word_count", "trend_direction", "is_declining_label"]
df[cols_to_show].head(8)

Shape: 30000 rows, 45 columns
Unique content_id count: 30000  <- confirms one row = one page
Declining rate (target balance): 54.2%



,content_id,impressions_90d,days_since_last_update,avg_position,ctr,word_count,trend_direction,is_declining_label
0,content_304f48230142,3803,20,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,15320,25,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,12581,20,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,11751,22,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,19140,14,44.0,0.13,2803.0,down,1
5,content_d4084a4bc775,3970,20,8.5,0.03,3080.0,down,1
6,content_9a34b442b552,20,20,7.0,0.00,3059.0,down,1
7,content_a63219c6e95a,1724,22,21.2,0.06,NaN,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
A fixed if-statement rule can only combine 2-3 conditions before it becomes unreadable and
brittle. But "declining" here isn't driven by one clean threshold — it's a mix of weak
signals (staleness, position, CTR, word count, age) that each shift the odds a little, and
interact with each other differently across content types. Below I show that individual
signals correlate only weakly with the label on their own -- which is exactly why a model
that can combine several weak signals outperforms a single hand-written rule.

In [5]:
import numpy as np

# Individual signal correlations with the label -- each one alone is weak
signals = ["days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count", "content_age_days"]
for s in signals:
    corr = df[s].corr(df["is_declining_label"])
    print(f"{s:28s} corr with is_declining_label: {corr:+.3f}")

print("\n-> No single signal is strongly correlated on its own -- weak, scattered signal")
print("   is exactly the case where a model that weighs several features together beats")
print("   a rule built on one or two hard-coded thresholds.")

# Reference point from Week 1: the hand rule vs learned model gap already observed
print("\nFor comparison, from the starter pipeline (outputs/model_results.json):")
print("  Hand-written baseline  Precision@50: 0.240")
print("  Random forest          Precision@50: 0.740  (~3x lift)")

days_since_last_update       corr with is_declining_label: +0.081
impressions_90d              corr with is_declining_label: -0.018
avg_position                 corr with is_declining_label: -0.029
ctr                          corr with is_declining_label: -0.062
word_count                   corr with is_declining_label: +0.090
content_age_days             corr with is_declining_label: -0.164

-> No single signal is strongly correlated on its own -- weak, scattered signal
   is exactly the case where a model that weighs several features together beats
   a rule built on one or two hard-coded thresholds.

For comparison, from the starter pipeline (outputs/model_results.json):
  Hand-written baseline  Precision@50: 0.240
  Random forest          Precision@50: 0.740  (~3x lift)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.